In [9]:
import time
import subprocess
import os
from osgeo import gdal

In [11]:
Input_Directory  = 'C:/Users/jepir/Desktop/S2/Multiband/'
Output_Directory = 'C:/Users/jepir/Desktop/S2/Singleband/'
# S2 Features Codes
BANDS      = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12', 'NDVI', 'EVI', 'NDWI', 'NBR', 'GCVI', 'NDRE1', 'NDRE2', 'NDRE3', 'TCB', 'TCG', 'TCW']
GLCM       = ['CONTRAST', 'DISSIMILARITY', 'HOMOGENEITY', 'ASM', 'ENERGY', 'MAX', 'ENTROPY', 'MEAN', 'VARIANCE', 'CORRELATION']
GLCM_BANDS = [A + B for A in BANDS[0:10] for B in GLCM]

In [13]:
# Get file paths into dictionary with base folders as key
Files_Dictionary = {StudyArea_Code:
                    [os.path.join(Input_Directory, os.path.join(StudyArea_Code, File_Name)) for File_Name in os.listdir(os.path.join(Input_Directory, StudyArea_Code))]
                    for StudyArea_Code in os.listdir(Input_Directory)}

In [15]:
# Iterate all Study Areas in the Dictionary
for StudyArea in Files_Dictionary.keys():
    print(StudyArea)
    # Create directory to Save file if it not exists
    if not os.path.isdir(Output_Directory + StudyArea): 
        os.mkdir(Output_Directory + StudyArea)
    # Get Study Area paths
    for File in Files_Dictionary[StudyArea]:
        print(File.split('\\')[-1])
        # Get File type to select correct band codes to apply and get bands count
        File_Type = File.split('\\')[-1].split('_')[3]
        if File_Type == 'BANDS':
            Band_Codes = BANDS
            Band_Count = 21
        elif File_Type == 'GLCM':
            Band_Codes = GLCM_BANDS
            # Open Raster File and get bands count for splitting
            Raster     = gdal.Open(File)
            Band_Count = Raster.RasterCount
            Raster     = None
        # Iterate Bands for Splitting
        for i in range(1, Band_Count + 1):       
            Output_File_Path = Output_Directory + StudyArea + '/' + File.split('\\')[-1].replace(File_Type, Band_Codes[i-1])
            Command = 'gdal_translate -b ' + str(i) + ' -tr 25 25 -r cubic ' + File + ' ' + Output_File_Path
            subprocess.run(Command, check=True)
            #print(Command)
        print('----')

Extremadura_2017
Extremadura_2017_S2_BANDS_A.tif
----
Extremadura_2017_S2_BANDS_B.tif
----
Extremadura_2017_S2_BANDS_C.tif
----
Extremadura_2017_S2_BANDS_D.tif
----
HuescaZaragoza_2021
HuescaZaragoza_2021_S2_BANDS_A.tif
----
HuescaZaragoza_2021_S2_BANDS_B.tif
----
HuescaZaragoza_2021_S2_BANDS_C.tif
----
HuescaZaragoza_2021_S2_BANDS_D.tif
----
HuescaZaragoza_2021_S2_GLCM_A.tif
----
HuescaZaragoza_2021_S2_GLCM_B.tif
----
HuescaZaragoza_2021_S2_GLCM_C.tif
----
HuescaZaragoza_2021_S2_GLCM_D.tif
----
Leon_2019
Leon_2019_S2_BANDS_A.tif
----
Leon_2019_S2_BANDS_B.tif
----
Teruel_2022
Teruel_2022_S2_BANDS_A.tif
----
Teruel_2022_S2_GLCM_A.tif
----


In [31]:
len(GLCM_BANDS)

100

In [13]:
# Count time
Time_Process_Start = time.perf_counter()
# Get Raster files
Rasters = [Input_Directory + Path for Path in os.listdir(Input_Directory)]
# Iterate Rasters to split them
for Raster in Rasters:
    # Get Raster Name and Type
    Raster_Name = Raster.split('/')[-1].split('.')[0]
    Raster_Type = Raster_Name.split('_')[3]
    # Get Raster bands count
    DS               = gdal.Open(Raster)
    Raster_BandCount = DS.RasterCount
    DS = None
    # Print which raster is being processed
    print(Raster_Name)
    # Select the Correct dictionary 
    if Raster_Type == 'BIO':
        Bands_Names = BIO
    elif Raster_Type == 'REFL':
        Bands_Names = REFL     
    elif 'GLCM' in Raster_Type:
        # Generate band names for the GLCM features for each band
        Band_Prefixes = [Raster_Type[4:7], Raster_Type[7:10]]
        Bands_Names   = [f"{BP}{Band}" for BP in Band_Prefixes for Band in GLCM]
    # Iterate Bands for Splitting
    for i in range(1, Raster_BandCount + 1):       
        Output_File_Path = Output_Directory + Raster_Name.replace(Raster_Type, Bands_Names[i-1]) + '.tif'
        print (Output_File_Path)
        if 'REMOVE' in Bands_Names[i-1]:
            continue
        Command = 'gdal_translate -b ' + str(i) + ' -tr 25 25 -r cubic ' + Raster + ' ' + Output_File_Path
        subprocess.run(Command, check=True)
    print('----')
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Extremadura_20190000_S2_BIO_A
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_LAI_A.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_FAPAR_A.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_FCOVER_A.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_CAB_A.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_CWC_A.tif
----
Extremadura_20190000_S2_BIO_B
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_LAI_B.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_FAPAR_B.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_FCOVER_B.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_CAB_B.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_CWC_B.tif
----
Extremadura_20190000_S2_BIO_C
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_LAI_C.tif
E:/Data/3_Data/Extremadura_2019/S2-L2A/Extremadura_20190000_S2_FAPAR_C.tif
E:/Data/3_Da